# Feature Engineering

## Race-Level Features

## Driver Season Features

## Constructor Features

## Pit Stop Features

In [32]:
# ============================================================
# NOTEBOOK 2: Feature Engineering
# ============================================================
# Goal: Create meaningful derived columns that help answer
# business questions about F1 performance.
# ============================================================

import pandas as pd
import numpy as np

# Load the cleaned master data
master = pd.read_csv("/content/drive/MyDrive/f1-analytics/data/processed/processedmaster_df.csv")
pit_stops = pd.read_csv("/content/drive/MyDrive/f1-analytics/data/processed/processedpit_stops_clean.csv")
print(f"Loaded master: {master.shape}")
print(master.head(2))

Loaded master: (26759, 24)
   resultId  raceId  driverId  constructorId  grid  position  positionOrder  \
0         1      18         1              1     1       1.0              1   
1         2      18         2              2     5       2.0              2   

   points  laps  statusId  ...              race_name        date  driver_ref  \
0    10.0    58         1  ...  Australian Grand Prix  2008-03-16    hamilton   
1     8.0    58         1  ...  Australian Grand Prix  2008-03-16    heidfeld   

   forename   surname driver_nationality         dob constructor_name  \
0     Lewis  Hamilton            British  1985-01-07          McLaren   
1      Nick  Heidfeld             German  1977-05-10       BMW Sauber   

  constructor_nationality     driver_name  
0                 British  Lewis Hamilton  
1                  German   Nick Heidfeld  

[2 rows x 24 columns]


In [33]:
# ── FEATURE 1: is_winner ─────────────────────────────────────
#
# Business meaning: Did this driver WIN this race?
# Why it matters: Win rate is the most fundamental F1 metric.
# Recruiter value: Shows you can create binary classification
#                  labels from raw data.
#
# positionOrder == 1 means first place.
# We use positionOrder (not position) because position has NaN
# for DNFs, but positionOrder always has a number.

master['is_winner'] = (master['positionOrder'] == 1).astype(int)
# .astype(int) converts True/False to 1/0

print("Win rate check:")
print(master['is_winner'].value_counts())

Win rate check:
is_winner
0    25631
1     1128
Name: count, dtype: int64


In [34]:
# ── FEATURE 2: podium_finish ─────────────────────────────────
#
# Business meaning: Did the driver finish in the top 3?
# Why it matters: Podiums measure consistent excellence,
#                 not just lucky wins.

master['podium_finish'] = (master['positionOrder'] <= 3).astype(int)

print("Podium distribution:")
print(master['podium_finish'].value_counts())

Podium distribution:
podium_finish
0    23362
1     3397
Name: count, dtype: int64


In [35]:
# ── FEATURE 3: top10_finish ──────────────────────────────────
#
# Business meaning: Did the driver finish in points (top 10)?
# Why it matters: Pre-2010, only top 8 scored. But top 10 is
#                 the modern points boundary. This measures
#                 "did they score points?" — critical for
#                 constructor championship math.

master['top10_finish'] = (master['positionOrder'] <= 10).astype(int)

In [36]:
# ── FEATURE 4: positions_gained ──────────────────────────────
#
# Business meaning: How many places did the driver gain/lose
#                   from their starting grid position to finish?
# Why it matters: A driver starting 15th and finishing 5th
#                 gained 10 positions — this measures racecraft
#                 and overtaking ability INDEPENDENT of the car.
#                 This is arguably the most interesting insight.
#
# Formula: grid - positionOrder
# Positive = moved FORWARD (gained positions)
# Negative = moved BACKWARD (lost positions)
# Zero = finished exactly where they started
#
# We must handle edge cases:
# - grid == 0 means pit lane start → exclude from calculation
# - NaN in either column → result is NaN (handled by pandas)

master['positions_gained'] = np.where(
    master['grid'] > 0,                             # condition: valid grid position
    master['grid'] - master['positionOrder'],       # true: calculate gain
    np.nan                                          # false: pit lane start → NaN
)

print("Positions gained distribution:")
print(master['positions_gained'].describe())
print("\nTop 5 single-race overtaking performances:")
print(master[['driver_name', 'race_name', 'year', 'grid', 'positionOrder',
              'positions_gained']].nlargest(5, 'positions_gained'))

Positions gained distribution:
count    25121.000000
mean         0.002189
std          7.185223
min        -31.000000
25%         -4.000000
50%          1.000000
75%          4.000000
max         30.000000
Name: positions_gained, dtype: float64

Top 5 single-race overtaking performances:
          driver_name           race_name  year  grid  positionOrder  \
18758    Jim Rathmann    Indianapolis 500  1957    32              2   
19103  Johnny Thomson    Indianapolis 500  1955    33              4   
19874     Andy Linden    Indianapolis 500  1951    31              4   
19305  Roberto Mieres  British Grand Prix  1954    32              6   
19302  Onofre Marimón  British Grand Prix  1954    28              3   

       positions_gained  
18758              30.0  
19103              29.0  
19874              27.0  
19305              26.0  
19302              25.0  


In [37]:
# ── FEATURE 5: did_finish ────────────────────────────────────
#
# Business meaning: Did the driver finish the race (not retire)?
# Why it matters: Reliability is a huge factor in championship
#                 success. A car that retires often can't win
#                 a title even with fast race pace.
#
# In this dataset, drivers who retired still have a positionOrder
# (their classified position), but their 'laps' completed is much
# less than the total race laps. A simpler proxy: if statusId == 1,
# the driver finished normally.
#
# Note: statusId 1 = "Finished" in the F1 database.

master['did_finish'] = (master['statusId'] == 1).astype(int)

In [38]:
# ── FEATURE 6: SEASON-LEVEL DRIVER STATS ─────────────────────
#
# Now we aggregate to the season level. This is useful for
# comparing drivers across full seasons, not just single races.

driver_season_stats = master.groupby(['driver_name', 'year']).agg(
    races_entered   = ('resultId', 'count'),     # total races in season
    total_wins      = ('is_winner', 'sum'),       # race wins
    total_podiums   = ('podium_finish', 'sum'),   # podium finishes
    total_points    = ('points', 'sum'),          # total points
    total_top10     = ('top10_finish', 'sum'),    # points finishes
    avg_position    = ('positionOrder', 'mean'),  # average finish position
    avg_grid        = ('grid', 'mean'),           # average starting position
    avg_pos_gained  = ('positions_gained', 'mean'),  # avg overtaking performance
    total_dnf       = ('did_finish', lambda x: (x == 0).sum())  # total DNFs
).reset_index()

# Add win rate: what % of entered races did they win?
driver_season_stats['win_rate'] = (
    driver_season_stats['total_wins'] / driver_season_stats['races_entered'] * 100
).round(2)

# Add podium rate
driver_season_stats['podium_rate'] = (
    driver_season_stats['total_podiums'] / driver_season_stats['races_entered'] * 100
).round(2)

print("Driver season stats sample:")
print(driver_season_stats.head())
print(f"\nShape: {driver_season_stats.shape}")

Driver season stats sample:
    driver_name  year  races_entered  total_wins  total_podiums  total_points  \
0  Adolf Brudes  1952              1           0              0           0.0   
1   Adolfo Cruz  1953              1           0              0           0.0   
2  Adrian Sutil  2007             17           0              0           1.0   
3  Adrian Sutil  2008             18           0              0           0.0   
4  Adrian Sutil  2009             17           0              0           5.0   

   total_top10  avg_position   avg_grid  avg_pos_gained  total_dnf  win_rate  \
0            0     16.000000  19.000000        3.000000          1       0.0   
1            0     16.000000  13.000000       -3.000000          1       0.0   
2            1     17.117647  20.294118        3.176471         16       0.0   
3            0     17.722222  19.166667        1.444444         17       0.0   
4            3     14.823529  13.705882       -1.117647         11       0.0   

   p

In [39]:
# ── FEATURE 7: CONSTRUCTOR SEASON STATS ──────────────────────
#
# Same idea but for teams. Teams have 2 drivers per race,
# so their stats are aggregated across both.

constructor_season_stats = master.groupby(['constructor_name', 'year']).agg(
    total_wins    = ('is_winner', 'sum'),
    total_podiums = ('podium_finish', 'sum'),
    total_points  = ('points', 'sum'),
    total_races   = ('resultId', 'count'),
    avg_position  = ('positionOrder', 'mean'),
    total_dnf     = ('did_finish', lambda x: (x == 0).sum())
).reset_index()

constructor_season_stats['win_rate'] = (
    constructor_season_stats['total_wins'] /
    constructor_season_stats['total_races'] * 100
).round(2)

print("Constructor season stats sample:")
print(constructor_season_stats.head())

Constructor season stats sample:
  constructor_name  year  total_wins  total_podiums  total_points  \
0              AFM  1952           0              0           0.0   
1              AFM  1953           0              0           0.0   
2              AGS  1986           0              0           0.0   
3              AGS  1987           0              0           1.0   
4              AGS  1988           0              0           0.0   

   total_races  avg_position  total_dnf  win_rate  
0            3     13.666667          3       0.0  
1            4     25.750000          4       0.0  
2            2     21.000000          2       0.0  
3           14     14.642857         14       0.0  
4           16     16.750000         16       0.0  


In [40]:
# ── FEATURE 8: PIT STOP AGGREGATES ───────────────────────────
#
# For each race, aggregate each driver's pit stop data
# into summary features we can merge with the master table.

pit_agg = pit_stops.groupby(['raceId', 'driverId']).agg(
    total_stops     = ('stop', 'count'),         # how many stops
    avg_stop_time   = ('duration', 'mean'),      # average pit duration
    min_stop_time   = ('duration', 'min'),       # fastest stop
    total_pit_time  = ('duration', 'sum')        # total time lost in pits
).reset_index()

# Round for cleanliness
pit_agg['avg_stop_time']  = pit_agg['avg_stop_time'].round(3)
pit_agg['min_stop_time']  = pit_agg['min_stop_time'].round(3)
pit_agg['total_pit_time'] = pit_agg['total_pit_time'].round(3)

print("Pit stop aggregates:")
print(pit_agg.head())

Pit stop aggregates:
   raceId  driverId  total_stops  avg_stop_time  min_stop_time  total_pit_time
0     841         1            2         23.213         23.199          46.426
1     841         2            2         24.046         22.994          48.092
2     841         3            1         23.716         23.716          23.716
3     841         4            3         24.055         23.251          72.165
4     841         5            1         24.865         24.865          24.865


In [41]:
# ── SAVE ALL ENGINEERED FEATURES ─────────────────────────────

processed_path = ("/content/drive/MyDrive/f1-analytics/data/processed")

# Merge pit stop aggregates into master
master_with_pits = pd.merge(
    master,
    pit_agg,
    on=['raceId', 'driverId'],
    how='left'
)

master_with_pits.to_csv("/content/drive/MyDrive/f1-analytics/data/processed" + "master_df.csv", index=False)
driver_season_stats.to_csv("/content/drive/MyDrive/f1-analytics/data/processed" + "driver_season_stats.csv", index=False)
constructor_season_stats.to_csv("/content/drive/MyDrive/f1-analytics/data/processed" + "constructor_season_stats.csv", index=False)
pit_agg.to_csv(processed_path + "pit_agg.csv", index=False)

print("✅ All engineered features saved!")
print(f"master_df columns: {master_with_pits.columns.tolist()}")

✅ All engineered features saved!
master_df columns: ['resultId', 'raceId', 'driverId', 'constructorId', 'grid', 'position', 'positionOrder', 'points', 'laps', 'statusId', 'fastestLapTime', 'fastestLapSpeed', 'year', 'round', 'race_name', 'date', 'driver_ref', 'forename', 'surname', 'driver_nationality', 'dob', 'constructor_name', 'constructor_nationality', 'driver_name', 'is_winner', 'podium_finish', 'top10_finish', 'positions_gained', 'did_finish', 'total_stops', 'avg_stop_time', 'min_stop_time', 'total_pit_time']
